# Examen 2
> Moctezuma Ramirez Diego Rafael

## Carga de modulos

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
import pandas as pd
import numpy as np

import plotly.express as px

# Data preprocessing
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling
from sklearn.model_selection import cross_val_score, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, Lars, Lasso, BayesianRidge, SGDRegressor

#pd.set_option('display.float_format', lambda x: '%.6f' % x)

## Funciones auxiliares

In [3]:
def frecuencias(df, caracteristicas):
    for caracteristica in caracteristicas:
        print(f"Caracteristica: {caracteristica}")
        abs_ = df[caracteristica].value_counts(dropna=False).to_frame().rename(columns={"count": "Frecuencia absoluta"})
        rel_ = df[caracteristica].value_counts(dropna=False, normalize= True).to_frame().rename(columns={"proportion": "Frecuencia relativa"})
        freq = abs_.join(rel_)
        freq["Frecuencia Acumulada"] = freq["Frecuencia absoluta"].cumsum()
        freq["Acumulada %"] = freq["Frecuencia relativa"].cumsum()
        freq["Frecuencia absoluta"] = freq["Frecuencia absoluta"].map(lambda x: f"{x:,.0f}")
        freq["Frecuencia relativa"] = freq["Frecuencia relativa"].map(lambda x: f"{x:,.2%}")
        freq["Frecuencia Acumulada"] = freq["Frecuencia Acumulada"].map(lambda x: f"{x:,.0f}")
        freq["Acumulada %"] = freq["Acumulada %"].map(lambda x: f"{x:,.2%}")
        display(freq)

## Carga de datos con pandas

In [4]:
df = pd.read_csv("DataRegresion/train_PAY_AMT2.csv",sep='|')

In [5]:
df.sample(5)

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
581,21261,230000.0,2,3,1,30,0,0,0,0,238601.0,135099.0,133431.0,53103.0,49586.0,3826.0,5000.0,5000.0,2000.0,3000.0
3065,7580,20000.0,1,2,2,27,-2,-2,-2,-2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2223,22051,200000.0,1,2,1,31,-2,-2,-2,-2,1747.0,642.0,0.0,0.0,0.0,642.0,0.0,0.0,0.0,500.0
1047,14325,300000.0,2,2,2,41,-1,-1,-1,-1,43909.0,27785.0,5221.0,1278.0,3245.0,27863.0,5234.0,1278.0,3245.0,22.0
4930,323,60000.0,2,1,2,24,0,2,0,0,55760.0,59007.0,56492.0,57565.0,56569.0,4800.0,0.0,2100.0,2200.0,2500.0


In [6]:
df.dtypes

CUSTOMER_ID      int64
LIMIT_BAL      float64
SEX              int64
EDUCATION        int64
MARRIAGE         int64
AGE              int64
PAY_3            int64
PAY_4            int64
PAY_5            int64
PAY_6            int64
BILL_AMT2      float64
BILL_AMT3      float64
BILL_AMT4      float64
BILL_AMT5      float64
BILL_AMT6      float64
PAY_AMT2       float64
PAY_AMT3       float64
PAY_AMT4       float64
PAY_AMT5       float64
PAY_AMT6       float64
dtype: object

In [7]:
df.isna().sum()

CUSTOMER_ID    0
LIMIT_BAL      0
SEX            0
EDUCATION      0
MARRIAGE       0
AGE            0
PAY_3          0
PAY_4          0
PAY_5          0
PAY_6          0
BILL_AMT2      0
BILL_AMT3      0
BILL_AMT4      0
BILL_AMT5      0
BILL_AMT6      0
PAY_AMT2       0
PAY_AMT3       0
PAY_AMT4       0
PAY_AMT5       0
PAY_AMT6       0
dtype: int64

## EDA

### Filtrado de variables

In [8]:
ls_discretas = ["SEX","EDUCATION","MARRIAGE","PAY_3","PAY_4","PAY_5","PAY_6"]
ls_continuas = ["BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6","LIMIT_BAL","AGE"]
ls_drop = ["CUSTOMER_ID"]

target = "PAY_AMT2"

In [9]:
df_customer_ids = df["CUSTOMER_ID"]
df.drop(columns=ls_drop, inplace=True)

### EDA Discreto

In [10]:
frecuencias(df, ls_discretas)

Caracteristica: SEX


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
SEX,,,,
2,"3,398",60.41%,"3,398",60.41%
1,"2,227",39.59%,"5,625",100.00%


Caracteristica: EDUCATION


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
EDUCATION,,,,
2,"2,627",46.70%,"2,627",46.70%
1,"1,960",34.84%,"4,587",81.55%
3,938,16.68%,"5,525",98.22%
5,55,0.98%,"5,580",99.20%
4,28,0.50%,"5,608",99.70%
6,14,0.25%,"5,622",99.95%
0,3,0.05%,"5,625",100.00%


Caracteristica: MARRIAGE


...

,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_4,,,,
0,"3,052",54.26%,"3,052",54.26%
-1,"1,082",19.24%,"4,134",73.49%
-2,826,14.68%,"4,960",88.18%
2,595,10.58%,"5,555",98.76%
3,39,0.69%,"5,594",99.45%
4,16,0.28%,"5,610",99.73%
7,8,0.14%,"5,618",99.88%
5,6,0.11%,"5,624",99.98%
8,1,0.02%,"5,625",100.00%


Caracteristica: PAY_5


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_5,,,,
0,"3,177",56.48%,"3,177",56.48%
-1,"1,024",18.20%,"4,201",74.68%
-2,873,15.52%,"5,074",90.20%
2,485,8.62%,"5,559",98.83%
3,36,0.64%,"5,595",99.47%
4,17,0.30%,"5,612",99.77%
7,8,0.14%,"5,620",99.91%
5,3,0.05%,"5,623",99.96%
8,1,0.02%,"5,624",99.98%


Caracteristica: PAY_6


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_6,,,,
0,"3,070",54.58%,"3,070",54.58%
-1,"1,032",18.35%,"4,102",72.92%
-2,940,16.71%,"5,042",89.64%
2,522,9.28%,"5,564",98.92%
3,39,0.69%,"5,603",99.61%
4,10,0.18%,"5,613",99.79%
7,5,0.09%,"5,618",99.88%
6,4,0.07%,"5,622",99.95%
5,2,0.04%,"5,624",99.98%


### EDA Continuas

In [11]:
for caracteristica in ls_continuas:
    fig = px.histogram(df, x=caracteristica, nbins=50, title=f"Histograma de {caracteristica}",template = "plotly_dark")
    fig.show()

### Feature Engineering

In [12]:
df['TOTAL_DEBT'] = df[['BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].sum(axis=1)
df['USAGE_RATIO'] = df['TOTAL_DEBT'] / df['LIMIT_BAL']
df['TOTAL_PAYMENTS'] = df[['PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].sum(axis=1)
df['PAY_AMT_MEAN'] = df[['PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].mean(axis=1)
df['PAY_STA_MEAN'] = df[['PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].mean(axis=1).round(2)

#### Normalización

> EDUCATION

In [13]:
redundantes_e = [0,5,6]
df['EDUCATION'] = df['EDUCATION'].replace(redundantes_e, 4)

In [14]:
df["EDUCATION"].value_counts(True)

EDUCATION
2    0.467022
1    0.348444
3    0.166756
4    0.017778
Name: proportion, dtype: float64

> MARRIAGE

In [15]:
redundantes_m = [0]
df['MARRIAGE'] = df['MARRIAGE'].replace(redundantes_m, 3)

In [16]:
df["MARRIAGE"].value_counts(True)

MARRIAGE
2    0.523022
1    0.465600
3    0.011378
Name: proportion, dtype: float64

In [17]:
df[ls_discretas].sample(5)

,SEX,EDUCATION,MARRIAGE,PAY_3,PAY_4,PAY_5,PAY_6
2850,2,1,2,-2,-2,-2,-2
3911,2,1,2,0,0,0,0
3045,2,2,2,-2,-1,0,0
453,1,1,2,2,2,2,2
3037,1,2,1,0,0,0,0


### Data Cleaning

In [18]:
shape_out = df.shape

In [19]:
df[ls_continuas].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,LIMIT_BAL,AGE
count,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000,5625.000000
mean,48406.229867,46197.471467,42512.594311,39937.451378,38295.546311,5276.970667,4867.177067,4551.361422,4836.827733,167993.955556,35.537600
std,68742.250299,66284.469902,62001.903751,59240.723853,57967.407861,16368.810543,15293.878245,14314.152245,15927.656615,129925.695333,9.193726
min,-30000.000000,-34041.000000,-34503.000000,-81334.000000,-51443.000000,0.000000,0.000000,0.000000,0.000000,10000.000000,21.000000
1%,-189.080000,-182.600000,-202.280000,-200.000000,-200.000000,0.000000,0.000000,0.000000,0.000000,10000.000000,22.000000
5%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20000.000000,23.000000
50%,20696.000000,19993.000000,19005.000000,18028.000000,16775.000000,1773.000000,1500.000000,1500.000000,1445.000000,140000.000000,34.000000
95%,194511.200000,181599.800000,172120.400000,165775.000000,161819.400000,19500.800000,15308.800000,15000.000000,16005.600000,430000.000000,53.000000
99%,321012.560000,313674.640000,294688.120000,280507.800000,272035.520000,68561.160000,70000.000000,66476.080000,74357.840000,500000.000000,60.000000
max,624475.000000,632041.000000,542653.000000,547880.000000,474459.000000,397092.000000,292462.000000,331788.000000,290000.000000,750000.000000,75.000000


In [20]:
dic_outliers = {}
for caracteristica in ls_continuas:
    aux = df[caracteristica].quantile(0.99) + df[caracteristica].std() * 4
    max_val = df[caracteristica].max()
    if max_val > aux:
        dic_outliers[caracteristica] = df[caracteristica].quantile(0.99)

In [21]:
dic_outliers

{'BILL_AMT2': np.float64(321012.5600000001),
 'BILL_AMT3': np.float64(313674.6400000019),
 'BILL_AMT5': np.float64(280507.80000000034),
 'PAY_AMT3': np.float64(68561.16000000003),
 'PAY_AMT4': np.float64(70000.0),
 'PAY_AMT5': np.float64(66476.0800000005),
 'PAY_AMT6': np.float64(74357.84000000056)}

In [22]:
for caracteristica, perc in dic_outliers.items():
    df = df[(df[caracteristica] <= perc) | (df[caracteristica].isna())]

In [23]:
df.shape[0] / shape_out[0] 

0.9514666666666667

#### Variables altamente vacías

In [24]:
df.isna().mean().sort_values()

LIMIT_BAL         0.0
TOTAL_PAYMENTS    0.0
USAGE_RATIO       0.0
TOTAL_DEBT        0.0
PAY_AMT6          0.0
PAY_AMT5          0.0
PAY_AMT4          0.0
PAY_AMT3          0.0
PAY_AMT2          0.0
BILL_AMT6         0.0
BILL_AMT5         0.0
BILL_AMT4         0.0
BILL_AMT3         0.0
BILL_AMT2         0.0
PAY_6             0.0
PAY_5             0.0
PAY_4             0.0
PAY_3             0.0
AGE               0.0
MARRIAGE          0.0
EDUCATION         0.0
SEX               0.0
PAY_AMT_MEAN      0.0
PAY_STA_MEAN      0.0
dtype: float64

#### Variables unarias

In [25]:
df.nunique()

LIMIT_BAL           68
SEX                  2
EDUCATION            4
MARRIAGE             3
AGE                 53
PAY_3               11
PAY_4                9
PAY_5               10
PAY_6               10
BILL_AMT2         4510
BILL_AMT3         4484
BILL_AMT4         4394
BILL_AMT5         4295
BILL_AMT6         4185
PAY_AMT2          2261
PAY_AMT3          2082
PAY_AMT4          1978
PAY_AMT5          1884
PAY_AMT6          1871
TOTAL_DEBT        4959
USAGE_RATIO       5052
TOTAL_PAYMENTS    3597
PAY_AMT_MEAN      3597
PAY_STA_MEAN        32
dtype: int64

In [26]:
df.std()

LIMIT_BAL         123273.783287
SEX                    0.488895
EDUCATION              0.750627
MARRIAGE               0.520663
AGE                    9.238350
PAY_3                  1.214005
PAY_4                  1.180621
PAY_5                  1.146079
PAY_6                  1.164936
BILL_AMT2          56601.913615
BILL_AMT3          53990.089975
BILL_AMT4          50215.445508
BILL_AMT5          47281.520770
BILL_AMT6          46578.255489
PAY_AMT2           12762.424627
PAY_AMT3            6843.925822
PAY_AMT4            6704.620915
PAY_AMT5            6235.179727
PAY_AMT6            6456.920648
TOTAL_DEBT        246115.515122
USAGE_RATIO            1.758112
TOTAL_PAYMENTS     17884.280183
PAY_AMT_MEAN        4471.070046
PAY_STA_MEAN           1.064320
dtype: float64

### Selección mejores variables

In [27]:
kb = SelectKBest(k="all", score_func=f_regression)

In [28]:
columnas = df.columns.to_list()
columnas.remove(target)
X = df[columnas]
y = df[target]

In [29]:
kb.fit(X, y)

,score_func,<function f_r...t 0x1761f0160>
,k,'all'


In [30]:
df_scores = pd.DataFrame(data=zip(X.columns, kb.scores_), columns=["feature", "score"]).set_index("feature").sort_values(by="score", ascending=False)

In [31]:
df_scores.head(20)

,score
feature,
PAY_AMT_MEAN,641.542886
TOTAL_PAYMENTS,641.542886
BILL_AMT3,420.601885
PAY_AMT5,345.455787
PAY_AMT3,321.434820
PAY_AMT6,298.838067
BILL_AMT4,241.725324
LIMIT_BAL,232.700885
BILL_AMT5,204.723994


In [32]:
fig = px.bar(df_scores.head(20), template = "plotly_dark")
fig.show()

## Modelado

### Entrenamiento del modelo

#### Standard Escaler

In [33]:
ss = StandardScaler()

#### Regresión Lineal

In [34]:
lg = LinearRegression()

#### LARS

In [35]:
lrs = Lars()

#### LASSO

In [36]:
lasso = Lasso()

#### Ridge

In [37]:
ridge = Ridge()

#### Elastic Net

In [38]:
elan = ElasticNet()

#### Red Bayesiana

In [39]:
redbay = BayesianRidge()

#### Gradiente Estocástico Descendiente

In [40]:
ged = SGDRegressor()

### Comparación de modelos

#### Usando selectKBest

In [41]:
modelos = {
    "Regresion Lineal": lg,
    "Lars": lrs,
    "Lasso": lasso,
    "Ridge": ridge,
    "Elastic Net": elan,
    "Red Bayesiana": redbay,
    "Gradiente Estocástico": ged
}

In [42]:
df_kbest_model = pd.DataFrame(columns=["Regresion Lineal", "Lars", "Lasso", "Ridge", "Elastic Net", "Red Bayesiana", "Gradiente Estocástico"])
X_train, X_test, y_train, y_test = train_test_split(X, y)

y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [43]:
transformer = ColumnTransformer([
    ("scaler", ss, ls_continuas),
    ("dummies", OneHotEncoder(handle_unknown="ignore"), ls_discretas)
])

In [44]:
for k in range(5, 56, 5):
    resultados_k = {}   

    for nombre, modelo in modelos.items():
        pipe = Pipeline([
            ("prep", transformer),
            ("kbest", SelectKBest(f_regression, k=k)),
            ("modelo", modelo)
        ])
        pipe.fit(X_train, y_train)
        score = pipe.score(X_test, y_test)
        resultados_k[nombre] = score

    df_kbest_model.loc[k] = resultados_k

In [45]:
df_kbest_model

,Regresion Lineal,Lars,Lasso,Ridge,Elastic Net,Red Bayesiana,Gradiente Estocástico
5,0.109822,1.098217e-01,0.109830,0.109823,0.106269,0.109866,0.110798
10,0.169870,1.698702e-01,0.170146,0.170344,0.130299,0.171632,0.174734
15,0.397785,3.977848e-01,0.398000,0.398095,0.157358,0.398124,0.401800
20,0.405171,4.051707e-01,0.405342,0.405421,0.157743,0.405506,0.405748
25,0.407714,4.077139e-01,0.407882,0.408027,0.158029,0.408226,0.405814
30,0.408478,4.042038e-01,0.408572,0.408723,0.160063,0.408920,0.409574
35,0.410659,-2.161569e+00,0.410346,0.410499,0.160368,0.410585,0.407928
40,0.410993,1.965905e-01,0.410526,0.410622,0.160367,0.410733,0.409393
45,0.410937,-1.140253e+01,0.410496,0.410578,0.160367,0.410768,0.410345
50,0.410451,-5.606160e+32,0.410682,0.410606,0.160513,0.410850,0.413019


In [46]:
max_lr = [df_kbest_model["Regresion Lineal"].idxmax(),df_kbest_model["Regresion Lineal"].max()]
max_lars = [df_kbest_model["Lars"].idxmax(),df_kbest_model["Lars"].max()]
max_lasso = [df_kbest_model["Lasso"].idxmax(),df_kbest_model["Lasso"].max()]
max_ridge = [df_kbest_model["Ridge"].idxmax(),df_kbest_model["Ridge"].max()]
max_elnet = [df_kbest_model["Elastic Net"].idxmax(),df_kbest_model["Elastic Net"].max()]
max_redbay = [df_kbest_model["Red Bayesiana"].idxmax(),df_kbest_model["Red Bayesiana"].max()]
max_ged = [df_kbest_model["Gradiente Estocástico"].idxmax(),df_kbest_model["Gradiente Estocástico"].max()]

In [47]:
df_modelos = pd.DataFrame(columns=["modelo","mejor_k","mejor_score"])

df_modelos.loc[0] = ["Linear Regression", max_lr[0], max_lr[1]]
df_modelos.loc[1] = ["LARS", max_lars[0], max_lars[1]]
df_modelos.loc[2] = ["Lasso", max_lasso[0], max_lasso[1]]
df_modelos.loc[3] = ["Ridge", max_ridge[0], max_ridge[1]]
df_modelos.loc[4] = ["Elastic Net", max_elnet[0], max_elnet[1]]
df_modelos.loc[5] = ["Red Bayesiana", max_redbay[0], max_redbay[1]]
df_modelos.loc[6] = ["Gradiente Estocástico", max_ged[0], max_ged[1]]

In [48]:
df_modelos.sort_values(by="mejor_score", ascending=False)

,modelo,mejor_k,mejor_score
6,Gradiente Estocástico,50,0.413019
0,Linear Regression,40,0.410993
5,Red Bayesiana,55,0.410862
2,Lasso,50,0.410682
3,Ridge,40,0.410622
1,LARS,25,0.407714
4,Elastic Net,50,0.160513


##### Cross validation 

In [49]:
df_crossv = pd.DataFrame(columns = ["modelo","f1","f2","f3","f4","f5","mean","std"])

In [50]:
dic_cross = {
    "Regresion lineal": [lg, max_lr[0]],
    "LARS": [lrs, max_lars[0]],
    "Lasso": [lasso, max_lasso[0]],
    "Ridge": [ridge, max_ridge[0]],
    "Elastic Net": [elan, max_elnet[0]],
    "Red Bayesiana": [redbay, max_redbay[0]],
    "Gradiente Estocástico": [ged, max_ged[0]]
}

In [51]:
for nombre_modelo, info in dic_cross.items():
    pipe = Pipeline([
        ("prep", transformer),
        ("kbest", SelectKBest(score_func=f_regression, k=info[1])),
        ("modelo", info[0])
    ])

    cross = cross_val_score(estimator=pipe,X=X,y=y.values.ravel(),cv=5)

    df_crossv.loc[len(df_crossv)] = [nombre_modelo,cross[0],cross[1],cross[2],cross[3],cross[4],cross.mean(),cross.std()    ]

In [52]:
df_crossv

,modelo,f1,f2,f3,f4,f5,mean,std
0,Regresion lineal,0.529435,0.264862,0.390788,0.505931,0.366952,0.411593,0.096687
1,LARS,0.526300,0.261361,0.388822,0.504339,0.360785,0.408321,0.097340
2,Lasso,0.529234,0.265960,0.390571,0.505628,0.367137,0.411706,0.096239
3,Ridge,0.529008,0.267665,0.391152,0.504624,0.367795,0.412049,0.095385
4,Elastic Net,0.194803,0.178112,0.165553,0.169314,0.181702,0.177897,0.010260
5,Red Bayesiana,0.528430,0.269820,0.391125,0.502560,0.368171,0.412021,0.094158
6,Gradiente Estocástico,0.523834,0.270968,0.390382,0.503658,0.367037,0.411176,0.093042


##### Comparación de cross validations

In [53]:
df_crossv[["modelo","mean"]].sort_values(by="mean", ascending=False)

,modelo,mean
3,Ridge,0.412049
5,Red Bayesiana,0.412021
2,Lasso,0.411706
0,Regresion lineal,0.411593
6,Gradiente Estocástico,0.411176
1,LARS,0.408321
4,Elastic Net,0.177897


## Hiperparametrización

In [54]:
param_grids = {
    "Regresion lineal": {
        "modelo__fit_intercept": [True, False],
        "modelo__positive": [False, True]
    },

    "LARS": {
        "modelo__fit_intercept": [True, False],
        "modelo__n_nonzero_coefs": [5, 10, 20, 50, 100, np.inf]
    },

    "Lasso": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__max_iter": [1000, 5000, 10000]
    },

    "Ridge": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__solver": ["auto", "svd", "cholesky", "lsqr"]
    },

    "Elastic Net": {
        "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0],
        "modelo__l1_ratio": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        "modelo__max_iter": [1000, 5000, 10000]
    },

    "Red Bayesiana": {
        "modelo__alpha_1": [1e-6, 1e-4, 1e-2],
        "modelo__alpha_2": [1e-6, 1e-4, 1e-2],
        "modelo__lambda_1": [1e-6, 1e-4, 1e-2],
        "modelo__lambda_2": [1e-6, 1e-4, 1e-2]
    },

    "Gradiente Estocástico": {
        "modelo__loss": ["squared_error", "huber"],
        "modelo__penalty": ["l2", "l1", "elasticnet"],
        "modelo__alpha": [0.0001, 0.001, 0.01],
        "modelo__learning_rate": ["constant", "optimal", "invscaling", "adaptive"],
        "modelo__eta0": [0.001, 0.01, 0.1],
        "modelo__max_iter": [1000, 5000, 10000]
    }
}

In [55]:
df_hyper = pd.DataFrame(columns=["Modelo", "Mejor score", "Std", "Parametros buenos"])

for modelo, info in dic_cross.items():
    pipe = Pipeline([
        ("prep", transformer),
        ("kbest", SelectKBest(f_regression, k=info[1])),
        ("modelo", info[0])
    ])

    grid = param_grids[modelo]
    search = RandomizedSearchCV(estimator=pipe, param_distributions=grid, scoring="r2", n_iter=20, cv=5, n_jobs=-1, random_state=777)

    search.fit(X_train, y_train)
    df_hyper.loc[len(df_hyper)] = [modelo,search.best_score_,search.cv_results_['std_test_score'][search.best_index_],search.best_params_]

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning:

The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning:

The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.

/opt/anaconda3/envs/CD/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning:


10 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File

In [56]:
df_hyper.sort_values(by="Mejor score", ascending=False)

,Modelo,Mejor score,Std,Parametros buenos
4,Elastic Net,0.380696,0.177039,"{'modelo__max_iter': 1000, 'modelo__l1_ratio':..."
3,Ridge,0.370485,0.230358,"{'modelo__solver': 'lsqr', 'modelo__alpha': 10.0}"
1,LARS,0.366835,0.210811,"{'modelo__n_nonzero_coefs': 20, 'modelo__fit_i..."
6,Gradiente Estocástico,0.363189,0.246678,"{'modelo__penalty': 'l1', 'modelo__max_iter': ..."
2,Lasso,0.361272,0.258410,"{'modelo__max_iter': 1000, 'modelo__alpha': 10.0}"
5,Red Bayesiana,0.360857,0.255689,"{'modelo__lambda_2': 0.0001, 'modelo__lambda_1..."
0,Regresion lineal,0.358283,0.263492,"{'modelo__positive': False, 'modelo__fit_inter..."


## Predicción

### Mejor modelo

#### Standard - Select K Best

In [57]:
nombre_modelo = df_crossv.loc[df_crossv["mean"].idxmax(), "modelo"]
info_modelo = dic_cross[nombre_modelo]
nombre_modelo

'Ridge'

In [58]:
modelo_ss = info_modelo[0]
mejor_k = info_modelo[1]

In [59]:
pipe_ss = Pipeline([
    ("prep", transformer),
    ("kbest", SelectKBest(f_regression, k=mejor_k)),
    ("modelo", modelo_ss)
])

In [60]:
pipe_ss.fit(X, y)

,steps,"[('prep', ...), ('kbest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


#### Hiperparametrización

In [61]:
df_hyper["Parametros buenos"][df_hyper["Modelo"] == "Elastic Net"].values[0]

{'modelo__max_iter': 1000, 'modelo__l1_ratio': 0.0, 'modelo__alpha': 0.01}

In [62]:
modelo_h = ElasticNet(max_iter=1000,l1_ratio=0.4, alpha=0.01)
mejor_k = max_elnet[0]

In [63]:
pipe_h = Pipeline([
    ("prep", transformer),
    ("kbest", SelectKBest(f_regression, k=mejor_k)),
    ("modelo", modelo_h)
])

In [64]:
pipe_h.fit(X, y)

,steps,"[('prep', ...), ('kbest', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Carga de datos

In [65]:
df_predic = pd.read_csv("DataRegresion/val_PAY_AMT2.csv",sep='|')

In [66]:
df_predic

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,14763,130000.0,1,1,2,29,0,0,0,0,108204.0,99784.0,71451.0,67024.0,62655.0,4000.0,4000.0,7500.0,8200.0
1,20087,40000.0,1,3,2,23,0,0,0,-2,15899.0,16700.0,17500.0,0.0,0.0,1000.0,0.0,0.0,0.0
2,3337,60000.0,2,3,2,45,0,0,0,0,23908.0,24215.0,17728.0,18087.0,17730.0,1173.0,532.0,597.0,469.0
3,368,40000.0,1,2,1,55,-2,-2,-2,-2,-540.0,-930.0,-1320.0,-1710.0,-2100.0,0.0,0.0,0.0,0.0
4,29107,210000.0,1,2,2,33,-1,-1,-1,0,264.0,264.0,264.0,42515.0,43104.0,264.0,42515.0,1559.0,1472.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1870,8431,180000.0,2,2,2,35,-1,-1,-1,0,4194.0,1300.0,0.0,4689.0,4689.0,0.0,4689.0,0.0,1582.0
1871,21286,70000.0,2,1,2,28,-1,-1,-1,-1,39270.0,603.0,2288.0,1380.0,2028.0,2288.0,1380.0,2028.0,390.0
1872,25596,60000.0,2,2,1,42,0,2,2,2,58022.0,61331.0,55566.0,59331.0,60569.0,0.0,4675.0,2361.0,0.0
1873,23030,50000.0,2,1,2,25,3,2,2,2,43870.0,42891.0,43582.0,44561.0,45592.0,1700.0,2000.0,1900.0,1800.0


### Normalización

> EDUCATION

In [67]:
redundantes_e = [0,5,6]
df_predic['EDUCATION'] = df_predic['EDUCATION'].replace(redundantes_e, 4)

> MARRIAGE

In [68]:
redundantes_m = [0]
df_predic['MARRIAGE'] = df_predic['MARRIAGE'].replace(redundantes_m, 3)

### Limpieza

In [69]:
shape_viejo = df_predic.shape

In [70]:
dic_outliers

{'BILL_AMT2': np.float64(321012.5600000001),
 'BILL_AMT3': np.float64(313674.6400000019),
 'BILL_AMT5': np.float64(280507.80000000034),
 'PAY_AMT3': np.float64(68561.16000000003),
 'PAY_AMT4': np.float64(70000.0),
 'PAY_AMT5': np.float64(66476.0800000005),
 'PAY_AMT6': np.float64(74357.84000000056)}

In [71]:
for caracteristica, perc in dic_outliers.items():
    df_predic = df_predic[(df_predic[caracteristica] <= perc) | (df_predic[caracteristica].isna())]

In [72]:
df_predic.shape[0] / shape_viejo[0]

0.9504

### Predicción

In [73]:
ids = df_predic["CUSTOMER_ID"]
df_predic.drop(columns=ls_drop, inplace=True)

In [74]:
pred1 = pipe_ss.predict(df_predic)

In [75]:
pred2 = pipe_h.predict(df_predic)

In [76]:
df_resultado_cv = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred1
})

In [77]:
df_resultado_hp = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred2
})

In [82]:
df_resultado_cv.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_PAYAMT2.csv", index=False)

In [79]:
df_resultado_cv

,CARDHOLDER_ID,y_hat
0,14763,9206.217030
1,20087,3338.280946
2,3337,2430.749827
3,368,785.231025
4,29107,7181.204668
...,...,...
1870,8431,3194.043221
1871,21286,-9125.687837
1872,25596,4979.881916
1873,23030,990.382907


In [83]:
df_resultado_hp.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_PAYAMT2.csv", index=False)

In [81]:
df_resultado_hp

,CARDHOLDER_ID,y_hat
0,14763,8941.430813
1,20087,3248.293172
2,3337,2498.994663
3,368,682.240859
4,29107,7544.493050
...,...,...
1870,8431,3666.432571
1871,21286,-7567.293599
1872,25596,4264.106607
1873,23030,2346.166146
